# DdS Financial Statements — unified generator, 2022 → 2025
### Départ de Sentier · one notebook, all years

Produces the nine-tab workbook for **any** year from 2022 to 2025:

- **2022–2024** are re-derived from the published workbooks (whose income/expense sheets already
  contain the merged Wise + Stripe + Raiffeisen detail). Running these reproduces the published
  statements and validates the engine.
- **2025** is built from the four raw exports and adds the accrual layer — grants, payroll,
  receivables — which did not exist in earlier years.

Set `YEAR` in the config cell, or run the last cell to generate all four.

**Compliance basis** — Art. 69a ZGB applies the Code of Obligations to associations *mutatis
mutandis*: Art. 957a (complete, chronological, voucher per entry), Art. 958c (no offsetting,
consistency, clarity), Art. 958 (balance sheet, income statement, notes), Art. 958f (10-year retention).

*Operational summary of public Swiss sources, not legal advice. Confirm with the auditor before issue.*

## The processing rules

1. Wise is the base for income and expense flows.
2. Bulk `STRIPE` payout rows are **replaced** by the individual Stripe payments.
3. Stripe net = `Converted Amount − Converted Amount Refunded − Fee`.
4. Raiffeisen rows appended (CHF).
5. Foreign currency → EUR at the **annual-average** rate; balances at the **31 December** rate;
   the resulting **foreign-exchange difference** is shown separately.
6. **Nothing is deleted.** Every row carries `Included_in_summary` and `Exclusion_reason`.
7. Failed / cancelled transactions → `Included = No`.
8. Internal transfers between DdS's own accounts → `Included = No`.
9. **Refunds are contra-entries**: a refund of an expense is a negative entry in that *expense*
   category; a refund of income a negative in that *income* category. Never crosses sides.
10. **The equity chain must articulate.** Every statement opens at the **prior year's closing
    equity**, never at prior-year cash:

    ```
    opening equity + operating result + FX difference = closing equity
    closing equity (year N-1)                        = opening equity (year N)
    ```

    Checked in code. For a **new** statement a failure raises and the statement must not be issued.
    For an **approved** historical statement the break is reported but the statement is still
    generated, because approved figures are not restated — only the GA can undo a GA decision
    (Art. 65 ZGB).

**Rule 7 note:** `CANCELLED` was applied in the manual work but never documented. Without it the
2022 statement cannot be reproduced — four duplicate Hetzner charges of CHF 334.10 on 10 Oct 2022
would wrongly appear.

**Column-name gotcha:** 2022–2024 sheets use `Refined Category_XZ` (space); the 2025 files use
`Refined_Category_XZ` (underscore).

### Exchange rates

**2022-2024** use each published workbook's own rates (SNB monthly for 2022, SNB annual average for
2023 and 2024), so those years reproduce exactly.

**2025** uses a three-part method:

| Leg | Basis |
|---|---|
| foreign to EUR | the **actual rate booked on the transaction** (Wise records both legs; Stripe delivers EUR directly) |
| CHF-denominated transactions | the **CHF amount taken directly** - no rate applied at all |
| EUR to CHF | **SNB monthly average** for the month of the transaction |
| balances at 31.12 | **SNB rate at the reporting date** |

SERI accepts all of these (guidelines s.2.4: daily or monthly SNB, or the rate charged by the card
provider). An annual average is not on SERI's list, which is why the monthly basis is preferred.

Every ledger row now carries **both** an EUR and a CHF value, and the income statement sums the CHF
column rather than multiplying an EUR total by one blended rate.

**Enter the twelve monthly rates and the year-end rate in the config cell.** Until they are filled
the notebook falls back to the SNB annual average (0.93703, retrieved from the SNB data portal) and
the ECB 31.12 rate, and the  tab records which basis was actually used.

SNB monthly: https://data.snb.ch/en/topics/ziredev/cube/devkum

In [1]:
# ===================== CONFIG =====================
import openpyxl, pandas as pd, datetime, re
from openpyxl.styles import Font, PatternFill, Alignment

YEAR = 2025          # <-- set the reporting year: 2022, 2023, 2024 or 2025
PROJ, UP, OUT = "/mnt/project", "/mnt/user-data/uploads", "/mnt/user-data/outputs"

# ---------------------------------------------------------------------------
# EXCHANGE RATES
#
# 2022-2024: rates come from each published workbook's own 'exchange rate' tab
#            (SNB monthly for 2022, SNB annual average for 2023 and 2024).
#
# 2025 method:
#   leg 1  foreign -> EUR : the ACTUAL BOOKED RATE on the transaction
#                           (Wise gives both legs; Stripe delivers EUR directly).
#                           Where the transaction is CHF-denominated the CHF figure
#                           is a fact and no rate is applied at all.
#   leg 2  EUR -> CHF     : SNB MONTHLY average for the month of the transaction.
#                           SERI accepts this basis (guidelines s.2.4) and DdS used
#                           SNB monthly rates in 2022.
#   balances              : SNB rate at 31.12.2025.
#
# Source: https://data.snb.ch/en/topics/ziredev/cube/devkum  (Foreign exchange rates - Month)
#         https://data.snb.ch/en/topics/ziredev/cube/devkuu  (daily, for the year-end rate)
#
# >>> PASTE THE TWELVE MONTHLY SNB RATES (1 EUR = x CHF) AND THE YEAR-END RATE.
SNB_MONTHLY_EUR_CHF = {            # month -> 1 EUR = x CHF (SNB monthly average 2025)
    1:0.94563,  2:0.93762,  3:0.95353,  4:0.93879,  5:0.93326,  6:0.93546,
    7:0.93053,  8:0.93482,  9:0.93590, 10:0.92781, 11:0.93303, 12:0.93110,
}
# Other currencies, SNB monthly average 2025 (CHF per unit) - used where no booked rate exists
SNB_MONTHLY_OTHER = {
 "USD":{1:0.91075,2:0.90190,3:0.88070,4:0.82560,5:0.82400,6:0.79770,
        7:0.81290,8:0.80070,9:0.79610,10:0.80170,11:0.80635,12:0.79330},
 "CAD":{1:0.62899,2:0.62471,3:0.61386,4:0.59645,5:0.59632,6:0.58324,
        7:0.58736,8:0.58271,9:0.57232,10:0.57240,11:0.57434,12:0.57901},
 "SGD":{1:0.67174,2:0.66882,3:0.65665,4:0.63230,5:0.63831,6:0.62579,
        7:0.62748,8:0.62340,9:0.61761,10:0.61636,11:0.62118,12:0.61683},
}
SNB_YE_EUR_CHF   = 0.9311          # 1 EUR = x CHF at 31.12.2025
SNB_AVG_EUR_CHF  = 0.93703         # SNB annual average 2025 (retrieved from the SNB data portal)
SNB_AVG_USD_CHF  = 0.83065         # SNB annual average 2025

# Non-CHF, non-EUR currencies: used only where no booked rate is available.
# Expressed as units per 1 EUR (ECB annual average 2025).
UNITS_PER_EUR = {"EUR":1.0,"CHF":SNB_AVG_EUR_CHF,"USD":1.1300,"CAD":1.5788,"SGD":1.4757,"DKK":7.4634}
RATE_TO_EUR   = {k:(1/v if k!="CHF" else 1/SNB_AVG_EUR_CHF) for k,v in UNITS_PER_EUR.items()}

_missing_monthly = [m for m,v in SNB_MONTHLY_EUR_CHF.items() if v is None]
MONTHLY_OK = not _missing_monthly
YE_EUR_CHF = SNB_YE_EUR_CHF if SNB_YE_EUR_CHF else 0.9314   # ECB 31.12.2025 as interim

def eur_to_chf(date):
    """SNB monthly rate for the transaction month; annual average until the monthly
    figures are entered."""
    if MONTHLY_OK and date is not None and not pd.isna(date):
        m = pd.Timestamp(date).month
        return SNB_MONTHLY_EUR_CHF[m]
    return SNB_AVG_EUR_CHF

CFG = {
 2022: dict(mode="historical", file="2022_DdS_Financial_Balance_Sheet.xlsx",
      conv="Target amount (after fees) converted to CHF", ccy="CHF", e2c=1.0, raw=["Raiffreisen"],
      open_cash=[], close_cash=[("Wise",4210.04,4142.26),("Raiffeisen",None,3430.46)],
      receivable=[("Brightcon receivable",2511.50,2504.40)], liabilities=[],
      open_eq=0.00, close_eq=10077.11, fx=1545.81, founding=True),
 2023: dict(mode="historical", file="2023_DdS_Financial_Balance_Sheetupdated.xlsx",
      conv="Target amount (after fees) converted to EUR", ccy="EUR", e2c=1/1.030184, raw=["stripe_payments","Raiffreisen"],
      open_cash=[("Wise",4210.04,4142.26),("Raiffeisen",None,3430.46)],
      close_cash=[("Wise",7353.62,6824.89),("Raiffeisen",None,2226.63)],
      receivable=[], liabilities=[], open_eq=7572.72, close_eq=9051.52, fx=None, founding=False),
 2024: dict(mode="historical", file="2024_DdS_Financial_Balance_Sheetupdated_20260714.xlsx",
      conv="Target amount (after fees) converted to EUR", ccy="EUR", e2c=1/1.05, raw=["stripe_payments","Raiffeisen"],
      open_cash=[("Wise",7353.62,6824.89),("Raiffeisen",None,2226.63)],
      close_cash=[("Wise",3394.68,3197.45),("Raiffeisen",None,6201.57)],
      receivable=[], liabilities=[], open_eq=9051.52, close_eq=9399.02, fx=None, founding=False),
 2025: dict(mode="sources", ccy="EUR", e2c=SNB_AVG_EUR_CHF, ye=YE_EUR_CHF,
      open_cash=[("Wise",3394.68,3197.45),("Raiffeisen",None,6201.57)],
      close_cash=[("Wise",19324.31,None),("Wise SGD / CAD",0.0,0.00),("Raiffeisen",None,297730.58),("Stripe",0.0,0.00)],
      receivable=[], liabilities=[], open_eq=9399.02, close_eq=None, fx=None, founding=False),
}

GRP={2024:dict(income={"Brightcon":["Brightcon online","Brightcon physical","Brightcon sponsor"]},
      expense={"Spring school":["Spring school hotel","Spring school travel"],
       "Brightcon":["Brightcon catering","Brightcon food","Brightcon hotel","Brightcon online system",
                    "Brightcon social event","Brightcon travel","Brightcon tshirt"],
       "Autumn schools":["Autumn school hotel","Autumn school travel"]})}
ACTIVITIES=["Spring school","Summer school","Brightcon","Autumn school","Autumn schools"]
PROJECTS=["ADEME project","GreenGrocer project"]
RC={"travel refund","travel expense refund"}; NZ={"refunded later","refund"}
# 2025 accrual layer
GG_CASH=295800.00; GG_PM=0.50; GG_HR=41527.50/504
ACCRUALS=[("ACCRUAL-PAYROLL","Salary / payroll",41527.50,"CHF",
           "Employer cost Sep-Dec 2025 (gross 36,666.80 + SVA 2,956.30 + BVG 1,874.00 + UVG 30.40); paid Jan-Feb 2026"),
          ("ACCRUAL-CEA","Brightcon",5592.58,"EUR","CEA Grenoble - Brightcon 2025 venue and printing; paid 3 Mar 2026"),
          ("ACCRUAL-GIFT","Brightcon",61.71,"EUR","Brightcon 2025 cohost gift reimbursement; paid 20 Apr 2026")]
UVG_CORRECTION_2025=668.74  # EUR — Xiaojin's UVG (accident insurance) invoice covers all of 2026;
                            # ERD-2 wrongly included it at full value instead of the 2025-attributable
                            # portion. Corrected Sept 2026, before the receivable was claimed/paid.
ADEME_ELIGIBLE=123757.63-1667.98-333.60-UVG_CORRECTION_2025; ADEME_CASH=27363.00+41009.14
# Brightcon 2025 registrations invoiced in 2025 but settled in 2026 -> accrued income + receivable
BRIGHTCON_2026=[("2026-01-14",250.00),("2026-05-06",375.00),("2026-05-08",250.00)]
CLOSING={}   # rule 10: closing equity per year, populated as statements are generated

## Cell 4 — Ledger builders

Two functions turn very different inputs into one common shape. Everything downstream — rules,
summaries, workbook — works off that shape and never needs to know which year it is looking at.

### `build_historical(year)` — for 2022, 2023, 2024

Reads the `{year} income` and `{year} expense` sheets of a published workbook. Those sheets already
contain the merged Wise + Stripe + Raiffeisen detail and a `converted to CHF/EUR` column, so no
merging or conversion is needed — the function only applies the classification rules.

Note the column is `Refined Category_XZ` here, with a **space**; the 2025 files use an underscore.

### `build_sources(year)` — for 2025 onwards

Reads the four raw exports and does the real work:

| Step | What happens |
|---|---|
| **Wise incoming** | rows whose `Reference` contains `STRIPE` are flagged as bulk payouts and excluded — they are about to be replaced (rule 2). SBB refunds of ADEME travel are moved to the expense side as negatives (rule 9). |
| **Stripe** | each paid charge is added as income at `Converted Amount − Fee`. Partial refunds become separate negative contra-rows. Failed charges are kept but excluded (rule 7). |
| **Wise outgoing** | rows whose category contains "refund" have the word stripped, the sign flipped, and are booked to the **income** side — these are registration refunds, so they reduce income rather than adding expense (rule 9). |
| **Raiffeisen** | appended in CHF. The GreenGrocer receipt is recognised here and immediately flagged as deferred income. |

### `legs(r)` and per-row currency

`legs()` reads both sides of a Wise transaction and returns two things: the **EUR actually
debited** (the booked rate, a fact from the statement) and the **CHF amount** where the transaction
was CHF-denominated. `add()` then stores both an EUR and a CHF value on every row.

This is why the income statement can sum a CHF column rather than multiplying an EUR total by one
blended rate.

### What `add()` records

Each row carries: `ID` (the voucher reference), date, `Refined_Category_XZ` as entered,
the normalised `Category`, the original amount and currency, EUR, CHF, `Included_in_summary`,
`Exclusion_reason`, source account, reference and note.

**No row is ever dropped.** Exclusions are expressed by setting `Included_in_summary = No` with a
reason (rule 6). That is what keeps the ledger complete under Art. 957a.

In [2]:
# ===================== LEDGER =====================
def eur(a,c):
    try: return float(a)*RATE_TO_EUR.get(str(c).strip().upper(),1.0)
    except: return None
def canon(c):
    if c is None or (isinstance(c,float) and pd.isna(c)): return None
    s=str(c).strip()
    return {"office expense":"Office expense","summer school":"Summer school",
            "ademe project":"ADEME project","greengrocer project":"GreenGrocer project",
            "internal transfer":"Internal transfer"}.get(s.lower(),s)

def build_historical(year):
    c=CFG[year]; src=openpyxl.load_workbook(f"{PROJ}/{c['file']}",data_only=True); rows=[]
    for sheet,side in [(f"{year} income","income"),(f"{year} expense","expense")]:
        r0=list(src[sheet].iter_rows(values_only=True)); h=list(r0[0]); i={n:h.index(n) for n in h if n}
        for r in r0[1:]:
            v=r[i[c["conv"]]]
            if not isinstance(v,(int,float)): continue
            cat=str(r[i["Refined Category_XZ"]]).strip() if r[i["Refined Category_XZ"]] else "(blank)"
            st=str(r[i["Status"]] or "").upper(); note=r[i["Note"]] if "Note" in i else None; lc=cat.lower()
            def put(s,val,inc,rsn):
                rows.append(dict(Side=s,ID=r[i["ID"]],Date=r[i["Created on"]],
                  Refined_Category_XZ=r[i["Refined Category_XZ"]],Category=cat,
                  Amount=r[i.get("Target amount (after fees)")],Currency=r[i.get("Target currency")],
                  Value=round(val,2),CHF=round(val*CFG[year]["e2c"],2),
                  Included_in_summary="Yes" if inc else "No",
                  Exclusion_reason=rsn,Source="workbook",Reference=r[i.get("Reference")],Note=note))
            if "internal" in lc or (note and "internal" in str(note).lower()):
                put(side,v,False,"Internal transfer between own accounts (rule 8)"); continue
            if st in ("FAILED","CANCELLED"):
                put(side,v,False,f"Transaction did not complete (status {st}) (rule 7)"); continue
            if side=="income" and lc in RC:
                put("expense",-abs(v),True,"Refund of expense booked as negative contra-entry (rule 9)"); continue
            if st=="REFUNDED" or lc in NZ:
                put(side,v,False,"Charge refunded / matched pair — net zero"); continue
            put(side,v,True,"")
    return pd.DataFrame(rows), src

def build_sources(year):
    mi=pd.read_excel(f"{UP}/{year}_money_in.xlsx",0); mo=pd.read_excel(f"{UP}/{year}_money_out.xlsx",0)
    sp=pd.read_csv(f"{UP}/Stripe_payments_{year}.csv"); ra=pd.read_excel(f"{UP}/Raiffeisen_{year}.xlsx",0)
    for d in (mi,mo,sp): d["cat"]=d["Refined_Category_XZ"].map(canon)
    rows=[]
    def add(side,i,d,cat,amt,ccy,inc,rsn,srcname,ref=None,note=None,eur_booked=None,chf_direct=None,raw=None):
        """eur_booked  = EUR actually debited/credited (actual booked rate), if known
           chf_direct  = CHF amount, if the transaction is CHF-denominated"""
        cc=(str(ccy).upper() if ccy is not None and not (isinstance(ccy,float) and pd.isna(ccy)) else "EUR")
        e = eur_booked if eur_booked is not None else (eur(amt,ccy) if eur(amt,ccy) is not None else None)
        if chf_direct is not None: chf=chf_direct
        elif e is not None:       chf=e*eur_to_chf(d)
        else:                     chf=None
        rows.append(dict(Side=side,ID=i,Date=d,Refined_Category_XZ=(raw if raw is not None else cat),
            Category=cat,Amount=amt,Currency=cc,
            Value=round(e,2) if e is not None else None,
            CHF=round(chf,2) if chf is not None else None,
            Included_in_summary="Yes" if inc else "No",Exclusion_reason=rsn,Source=srcname,Reference=ref,Note=note))

    def legs(r):
        """Return (eur_booked, chf_direct) from a Wise row."""
        ta,tc=r.get("Target amount (after fees)"),str(r.get("Target currency") or "").upper()
        sa,sc=r.get("Source amount (after fees)"),str(r.get("Source currency") or "").upper()
        eb = float(sa) if sc=="EUR" and pd.notna(sa) else (float(ta) if tc=="EUR" and pd.notna(ta) else None)
        cd = float(ta) if tc=="CHF" and pd.notna(ta) else (float(sa) if sc=="CHF" and pd.notna(sa) else None)
        if cd is not None and eb is None: eb = cd/eur_to_chf(r.get("Created on"))
        return eb,cd
    no=lambda v:str(v).strip().lower()=="no"
    # Wise IN — rule 1,2
    for _,r in mi.iterrows():
        cat=r["cat"]; ref=str(r.get("Reference") or "")
        if "STRIPE" in ref.upper():
            add("income",r["ID"],r["Created on"],"Stripe bulk payout",r["Target amount (after fees)"],
                r["Target currency"],False,"Bulk Stripe payout replaced by individual payments (rule 2)","Wise",ref,
                raw=r.get("Refined_Category_XZ")); continue
        if no(r.get("Included")):
            add("income",r["ID"],r["Created on"],cat,r["Target amount (after fees)"],r["Target currency"],False,"Marked 'No' by treasurer","Wise",ref,raw=r.get("Refined_Category_XZ")); continue
        if cat and "internal" in str(cat).lower():
            add("income",r["ID"],r["Created on"],cat,r["Target amount (after fees)"],r["Target currency"],False,"Internal transfer between own accounts (rule 8)","Wise",ref,raw=r.get("Refined_Category_XZ")); continue
        if cat=="ADEME project" and str(r.get("Note") or "").lower().startswith("refund by sbb"):
            _e,_c=legs(r)
            add("expense",r["ID"],r["Created on"],cat,-abs(r["Target amount (after fees)"]),r["Target currency"],True,
                "Refund of expense booked as negative contra-entry (rule 9)","Wise",ref,r.get("Note"),
                -abs(_e) if _e is not None else None,-abs(_c) if _c is not None else None,
                raw=r.get("Refined_Category_XZ")); continue
        add("income",r["ID"],r["Created on"],cat,r["Target amount (after fees)"],r["Target currency"],True,"","Wise",ref,r.get("Note"),*legs(r),raw=r.get("Refined_Category_XZ"))
    # Stripe detail — rules 2,3
    for _,r in sp.iterrows():
        paid=str(r["Status"]).lower()=="paid"
        refund=r["Converted Amount Refunded"] if pd.notna(r["Converted Amount Refunded"]) else 0.0
        add("income",r["id"],r["Created date (UTC)"],r["cat"] if paid else "(uncategorised)",
            (r["Converted Amount"]-r["Fee"]) if paid else r["Converted Amount"],r["Converted Currency"],paid,
            "" if paid else f"Stripe status {r['Status']} (rule 7)","Stripe",None,r.get("Description"),
            raw=r.get("Refined_Category_XZ"))
        if paid and refund>0:
            add("income",str(r["id"])+"-REF",r["Refunded date (UTC)"] or r["Created date (UTC)"],r["cat"],
                -abs(refund),r["Converted Currency"],True,"Partial refund booked as negative contra-entry (rule 9)","Stripe",
                raw=r.get("Refined_Category_XZ"))
    # Wise OUT — rule 9
    RE=re.compile(r"\s*refunds?\s*",re.I)
    for _,r in mo.iterrows():
        cat=r["cat"]; ref=str(r.get("Reference") or "")
        if no(r.get("Included")):
            add("expense",r["ID"],r["Created on"],cat,r["Target amount (after fees)"],r["Target currency"],False,"Marked 'No' by treasurer","Wise",ref,raw=r.get("Refined_Category_XZ")); continue
        if cat and "internal" in str(cat).lower():
            add("expense",r["ID"],r["Created on"],cat,r["Target amount (after fees)"],r["Target currency"],False,"Internal transfer between own accounts (rule 8)","Wise",ref,raw=r.get("Refined_Category_XZ")); continue
        if cat and "refund" in str(cat).lower():
            _e,_c=legs(r)
            add("income",r["ID"],r["Created on"],canon(RE.sub(" ",str(cat)).strip()),-abs(r["Target amount (after fees)"]),
                r["Target currency"],True,"Refund of registration income booked as negative contra-entry (rule 9)","Wise",ref,None,
                -abs(_e) if _e is not None else None,-abs(_c) if _c is not None else None,
                raw=r.get("Refined_Category_XZ")); continue
        add("expense",r["ID"],r["Created on"],cat,r["Target amount (after fees)"],r["Target currency"],True,"","Wise",ref,r.get("Note"),*legs(r),raw=r.get("Refined_Category_XZ"))
    # Raiffeisen — rule 4
    ra=ra[ra["Credit/Debit Amount"].notna()].copy()
    for _,r in ra.iterrows():
        a=float(r["Credit/Debit Amount"]); cat=canon(r.get("Refined_Category_XZ")); t=str(r["Text"])
        if a==0:
            add("income",f"RAIFF-{r['Booked At']}",r["Booked At"],cat or "(closing entry)",0,"CHF",False,"Zero-value year-end closing entry","Raiffeisen",None,t,raw=r.get("Refined_Category_XZ")); continue
        gg=abs(a)==GG_CASH; internal=bool(cat and "internal" in str(cat).lower())
        add("income" if a>0 else "expense",f"RAIFF-{r['Booked At']}-{abs(a):.2f}",r["Booked At"],
            "GreenGrocer grant (deferred income)" if gg else cat,abs(a),"CHF",not(internal or gg),
            "Internal transfer between own accounts (rule 8)" if internal else
            ("SERI prefinancing received in advance; recognised as deferred income and released as eligible costs are incurred (contract 25.00414 s.3.1/3.4)" if gg else ""),
            "Raiffeisen",None,t,None,abs(a),raw=r.get("Refined_Category_XZ"))
    return pd.DataFrame(rows), (sp,ra)

## Cell 6 — Accruals and grants

This is the accrual layer, and it runs **only** when `mode == "sources"`. For 2022–2024 it returns
immediately, so those years stay on the cash basis their published statements used.

### Cost accruals

Three costs were incurred in 2025 but settled in 2026:

| Item | Amount | Paid |
|---|---|---|
| Payroll Sep–Dec (gross 36,666.80 + employer SVA 2,956.30 + BVG 1,874.00 + UVG 30.40) | CHF 41,527.50 | net 13 Jan 2026 |
| CEA Grenoble — Brightcon venue and printing | EUR 5,592.58 | 3 Mar 2026 |
| Brightcon cohost gift reimbursement | EUR 61.71 | 20 Apr 2026 |

Each becomes an expense row dated 31 December and a matching liability on the balance sheet.

### Income accrual

Three Brightcon 2025 registrations (EUR 250.00, 375.00, 250.00) were invoiced in 2025 and settled
in January and May 2026. The conference was held in October 2025, so the income belongs to 2025
with a receivable at the balance-sheet date.

> ⚠️ When these 2026 payments arrive they must be booked **against the receivable**, not recognised
> again as 2026 income. The same mistake across 2022 and 2023 double-counted CHF 2,504.40.

### GreenGrocer — deferred income

The contribution reimburses eligible costs and unused funds are repayable (contract §3.4), so income
is recognised only to the extent of costs incurred: personnel (`GG_PM` person-months), travel already
in the ledger, plus the **15% flat indirect rate**. The remainder of the CHF 295,800 becomes a
liability.

> ⚠️ **`GG_PM` is the largest single lever on the 2025 result.** It is set to 0.50 person-months
> (78.75 h at CHF 82.40/h). It must be evidenced
> by the timesheet.

### ADEME — accrued income

ADEME reimburses **70%** of eligible costs. The entitlement on costs certified in ERD-2 exceeds the
cash received, so the difference is recognised as income with a receivable. The 2024 Gothenburg
items are excluded from the base because DdS never bore them.

The function returns the extended ledger plus the liability and receivable lists the balance sheet
needs.

In [3]:
# ===================== ACCRUALS & GRANTS (2025 only) =====================
def apply_accruals(year, led):
    c=CFG[year]; extra=[]; liab=[]; recv=[]
    if c["mode"]!="sources": return led, liab, recv, {}
    E=c["e2c"]; YE=c["ye"]
    def row(side,i,cat,val,ccy,rsn,note=None):
        e=eur(val,ccy)
        chf = val if str(ccy).upper()=="CHF" else e*YE_EUR_CHF   # accrued at the reporting date
        extra.append(dict(Side=side,ID=i,Date=pd.Timestamp(f"{year}-12-31"),
            Refined_Category_XZ=cat,Category=cat,Amount=val,
            Currency=ccy,Value=round(e,2),CHF=round(chf,2),Included_in_summary="Yes",Exclusion_reason=rsn,
            Source="Accrual",Reference=None,Note=note))
    for i,cat,amt,ccy,note in ACCRUALS:
        row("expense",i,cat,amt,ccy,"Accrued 2025 cost, settled in 2026",note)
    liab += [("Accrued payroll Sep–Dec 2025",41527.50),
             ("Accrued expense — CEA (Brightcon venue)",5592.58*YE),
             ("Accrued expense — Brightcon cohost gift",61.71*YE)]
    # GreenGrocer: recognise income = eligible costs incurred (personnel + travel + 15% indirect)
    gg_travel_eur=led[(led.Side=="expense")&(led.Included_in_summary=="Yes")&
        (led.Category.astype(str).str.contains("GreenGrocer",case=False,na=False))]["Value"].sum()
    gg_pers=round(GG_PM*157.5*GG_HR,2)
    gg_total=round((gg_pers+gg_travel_eur*E)*1.15,2)
    row("income","GRANT-GG-2025","GreenGrocer project",gg_total,"CHF",
        f"Grant income recognised to match eligible costs incurred Sep–Dec 2025 ({GG_PM} PM personnel + travel + 15% flat indirect)")
    row("expense","ACCRUAL-GG-PERS","GreenGrocer project",gg_pers,"CHF",
        f"Personnel allocated to GreenGrocer ({GG_PM} PM = {GG_PM*157.5:.2f} h at CHF {GG_HR:.2f}/h); part of the payroll accrual")
    row("expense","ACCRUAL-PAYROLL-ADJ","Salary / payroll",-gg_pers,"CHF","Reallocation of payroll to GreenGrocer (contra)")
    liab.append(("Deferred project income — GreenGrocer (SERI 25.00414)",GG_CASH-gg_total))
    # Brightcon 2025 registrations received in 2026 -> income accrued to 2025, receivable at 31.12
    b_tot=round(sum(a for _,a in BRIGHTCON_2026),2)
    row("income","ACCRUED-BRIGHTCON","Brightcon",b_tot,"EUR",
        "Brightcon 2025 registration income invoiced in 2025 and received in 2026 ("
        +", ".join(f"EUR {a:,.2f} on {d}" for d,a in BRIGHTCON_2026)+")")
    recv.append(("Brightcon 2025 registrations receivable",b_tot,b_tot*YE))
    # ADEME: entitlement 70% of certified eligible costs, less cash received
    ent=round(ADEME_ELIGIBLE*0.70,2); acc=round(ent-ADEME_CASH,2)
    row("income","ACCRUED-ADEME","ADEME project",acc,"EUR",
        "Accrued ADEME income: 70% of eligible costs certified in ERD-2, less cash received")
    recv.append(("ADEME accrued income receivable",acc,acc*YE))
    return pd.concat([led,pd.DataFrame(extra)],ignore_index=True), liab, recv, dict(gg_total=gg_total,gg_def=GG_CASH-gg_total,ent=ent,acc=acc,brightcon=b_tot)

def grouped(s,g):
    o={};used=set()
    for k,cats in g.items(): o[k]=round(sum(s.get(x,0) for x in cats),2); used|=set(cats)
    for k,v in s.items():
        if k not in used: o[k]=round(o.get(k,0)+v,2)
    return o

## Cell 8 — Workbook writer

`write(year)` runs the whole pipeline for one year and saves the file. It is long because it
contains the presentation layer, but the sequence is simple.

### 1 · Build and classify
Calls the right ledger builder, applies the accrual layer, sorts chronologically (Art. 957a), and
groups by category into `IS`/`ES` (EUR) and `ISC`/`ESC` (CHF).

For 2024 a `GRP` mapping folds detail categories into the published presentation lines — the seven
`Brightcon *` expense categories become one *Brightcon* line, and so on.

### 2 · Compute the balance sheet
Cash at year-end rates, plus receivables, less liabilities, gives equity. The **foreign-exchange
difference** is derived so the statement articulates (§3.3 of the methodology document).

### 3 · Rule 10 — the equity check

```
opening equity + operating result + FX = closing equity
closing equity (year N−1)              = opening equity (year N)
```

For a **new** statement a failure raises and the file is not written. For an **approved** historical
statement the break is printed and the file still generates, because approved figures are not
restated — only the GA can undo a GA decision (Art. 65 ZGB).

### 4 · Write the nine tabs

`notes` · `exchange rate` · `{year} income` · `{year} income summary` · `{year} expense` ·
`{year} expense summary` · `stripe_payments` · `Raiffeisen` · `{year} summary`

The `notes` tab is generated conditionally: notes 1–4 appear every year, the founding-year note only
for 2022, and the accrual, grant, prior-period, accrued-income, net-assets and tax notes only where
they apply. Figures inside the notes are pulled from the calculation, so they cannot drift.

The `{year} summary` tab carries, in order: income statement · balance sheet · equity
reconciliation · cash and cash equivalents reconciliation · per-activity overview · notes.

### 5 · House formatting
`styled()` and `hdr()` apply the association's style to every sheet: Aptos Narrow, **gridlines
hidden**, 22-point section headings, bold headers with a thin rule, totals with rules above and
below, accounting number format. Excluded rows are greyed but remain fully legible.

In [4]:
# ===================== WORKBOOK =====================
def write(year):
    c=CFG[year]
    if c["mode"]=="historical": led,src=build_historical(year); raws=src
    else: led,raws=build_sources(year); src=None
    led,liab,recv,g = apply_accruals(year,led)
    led["Date"]=pd.to_datetime(led["Date"],errors="coerce",format="mixed",utc=True).dt.tz_localize(None)
    led=led.sort_values(["Side","Date"],na_position="last").reset_index(drop=True)
    inc_m=led[(led.Side=="income") &(led.Included_in_summary=="Yes")]
    exp_m=led[(led.Side=="expense")&(led.Included_in_summary=="Yes")]
    IS=inc_m.groupby("Category")["Value"].sum().round(2)
    ES=exp_m.groupby("Category")["Value"].sum().round(2)
    ISC=inc_m.groupby("Category")["CHF"].sum().round(2)
    ESC=exp_m.groupby("Category")["CHF"].sum().round(2)
    gg=GRP.get(year,{}); GI=grouped(IS,gg.get("income",{})); GE=grouped(ES,gg.get("expense",{}))
    GIC=grouped(ISC,gg.get("income",{})); GEC=grouped(ESC,gg.get("expense",{}))
    op=round(sum(GI.values())-sum(GE.values()),2)
    opc=round(sum(GIC.values())-sum(GEC.values()),2)   # CHF from per-row rates, not a blended total
    YE=c.get("ye",c["e2c"])
    close_cash=[(n,e,(e*YE if v is None and e is not None else v)) for n,e,v in c["close_cash"]]
    tot_cash=round(sum(v for _,_,v in close_cash if v),2)
    tot_recv=round(sum(v for _,_,v in (c["receivable"]+recv)),2)
    assets=round(tot_cash+tot_recv,2); tot_liab=round(sum(v for _,v in (c["liabilities"]+liab)),2)
    equity=round(assets-tot_liab,2)
    fx = c["fx"] if c["fx"] is not None else round(equity-c["open_eq"]-opc,2)
    wb=openpyxl.Workbook()
    # ---------- house style (matches the association's published workbooks) ----------
    from openpyxl.styles import Border, Side
    FONT="Aptos Narrow"; SZ=11
    BODY=Font(name=FONT,size=SZ); SEC=Font(name=FONT,size=SZ,bold=True)
    TITLE=Font(name=FONT,size=22,bold=True); ADDR=Font(name=FONT,size=12)
    THIN=Side(style="thin")
    B_UNDER=Border(bottom=THIN); B_TOTAL=Border(top=THIN,bottom=THIN)
    ACC='_(* #,##0.00_);_(* \\(#,##0.00\\);_(* "-"??_);_(@_)'
    GREY=PatternFill("solid",fgColor="F2F2F2"); W=Alignment(wrap_text=True,vertical="top")
    RIGHT=Alignment(horizontal="right")
    def styled(ws, ncols=None, widths=None):
        """Apply the house font, hide gridlines, format numbers."""
        ws.sheet_view.showGridLines=False
        for row in ws.iter_rows():
            for cell in row:
                f=cell.font
                cell.font=Font(name=FONT,size=22 if f.size==22 else SZ,bold=f.bold,color=f.color)
                if isinstance(cell.value,(int,float)) and not isinstance(cell.value,bool):
                    cell.number_format=ACC
        if widths:
            for col,w in widths.items(): ws.column_dimensions[col].width=w
    def hdr(ws,n,row=1):
        """Bold header row with a thin rule underneath (no fill — house style)."""
        for k in range(1,n+1):
            c=ws.cell(row=row,column=k); c.font=SEC; c.border=B_UNDER
            if k>1: c.alignment=RIGHT
    ws=wb.active; ws.title="notes"
    def put(cl,v,f=None):
        ws[cl]=v; ws[cl].alignment=W
        if f: ws[cl].font=f
    accr = c["mode"]=="sources"
    put("A1",f"Notes to the {year} Financial Statements",Font(bold=True,size=13))
    put("A3","1. Overview",SEC)
    put("A4",f"All financial transactions of Départ de Sentier (DdS) for 1 January – 31 December {year}, "
             "with the calculations used to prepare the summary.")
    put("A6","2. Accounts",SEC)
    put("A7","Wise (main account, EUR/multi-currency) · Raiffeisen (CHF) · Stripe (card registration income only). "
             "Transfers between them are internal: retained in the detail sheets, excluded from income and expense.")
    put("A9","3. Basis of preparation",SEC)
    put("A10","Prepared under Art. 69a of the Swiss Civil Code, which applies the Code of Obligations "
              "(Art. 957–958f) to associations mutatis mutandis.")
    put("A11","Income and expenses are recognised in the period in which they are earned or incurred, "
              "irrespective of when cash moves. From 2025 the accounts are prepared on an accrual basis "
              "because project grants and payroll made cash-basis reporting misleading; earlier years were "
              "prepared substantially on a cash basis. This change of presentation is disclosed here "
              "(Art. 958c CO, consistency)." if accr else
              "The association held no material payables at the balance-sheet date, so the accounts are "
              "prepared substantially on a cash basis. From 2025 an accrual basis is used.")
    put("A13","4. Reporting currency and foreign currency translation",SEC)
    put("A14",f"Presented in Swiss francs (CHF); EUR amounts reflect the association's principal working currency. "
              f"Income and expenses are translated at the {year} annual-average rate; cash balances at the "
              f"31 December {year} rate. The resulting foreign-exchange difference is recognised separately.")
    put("A16","5. Completeness and no offsetting (Art. 957a, 958c CO)",SEC)
    put("A17","Every transaction is retained in the detail sheets; none are deleted. Two columns document the "
              "treatment of each row: 'Included_in_summary' and 'Exclusion_reason'. Income and expenses are not "
              "offset; refunds are negative contra-entries within the original category.")
    put("A19","6. Treatment of specific items",SEC)
    put("A20","- Failed, cancelled and internal transactions: retained, excluded from the summary.")
    put("A21","- Bulk Stripe payouts are replaced by the individual card payments (net of Stripe fees and refunds).")
    r=22
    if accr:
        put(f"A{r}","- Costs relating to 2025 but settled in 2026 are accrued: payroll for September–December "
                    "(CHF 41,527.50), the CEA Brightcon venue invoice (EUR 5,592.58) and the Brightcon cohost "
                    "gift reimbursement (EUR 61.71)."); r+=2
        put(f"A{r}","7. Project grants",SEC); r+=1
        put(f"A{r}",f"GreenGrocer (SERI contract 25.00414, Horizon Europe 101182025): CHF {GG_CASH:,.2f} received "
              f"15 December {year} as the first of three instalments of a maximum CHF 591,481.00 for 01.09.2025–31.08.2029. "
              f"The contribution reimburses eligible costs and unused funds are repayable, so it is recognised as income "
              f"only to the extent of costs incurred. CHF {g['gg_total']:,.2f} is recognised in {year}; "
              f"CHF {g['gg_def']:,.2f} is carried as deferred income. In accordance with contract clause 3.2 the funds "
              f"are reported separately."); r+=2
        put(f"A{r}",f"ADEME (convention 2403D0042): the contribution reimburses 70% of eligible costs. Cumulative "
              f"eligible costs certified to 31 December {year} give an entitlement of EUR {g['ent']:,.2f}, against "
              f"EUR {ADEME_CASH:,.2f} received in cash, leaving EUR {g['acc']:,.2f} recognised as accrued income "
              f"receivable."); r+=2
    else: r+=1
    put(f"A{r}","8. Consistency and retention" if accr else "7. Consistency and retention",SEC); r+=1
    put(f"A{r}","Policies are applied consistently year to year; any change is disclosed here. Source exports, this "
                "workbook and the generating notebook are retained for 10 years (Art. 958f CO)."); r+=2
    put(f"A{r}",f"Generated {datetime.date.today().strftime('%-d %B %Y')}. Subject to review by the association's auditor before issue.")
    ws.column_dimensions["A"].width=118
    styled(ws)
    # exchange rate
    d=wb.create_sheet("exchange rate")
    if c["mode"]=="historical" and "exchange rate" in src.sheetnames:
        for row in src["exchange rate"].iter_rows(values_only=True): d.append(list(row))
        for col,w in zip("ABCDEF",[5,11,4,17,11,42]): d.column_dimensions[col].width=w
        styled(d)
    else:
        d.append(["Currency","Units per 1 EUR (annual average)","Units per 1 EUR (31.12)","Note"]); hdr(d,4)
        d.append(["Method","",""])
        d.append(["foreign -> EUR","actual rate booked on the transaction (Wise / Stripe)",""])
        d.append(["CHF transactions","CHF amount taken directly; no rate applied",""])
        d.append(["EUR -> CHF","SNB monthly average for the month of the transaction" if MONTHLY_OK
                  else "SNB annual average (monthly rates pending)",""])
        d.append(["balances at 31.12","SNB rate at the reporting date",""])
        d.append([])
        d.append(["Month","1 EUR = x CHF (SNB monthly average)",""])
        for m in range(1,13):
            d.append([f"{year}-{m:02d}",SNB_MONTHLY_EUR_CHF[m],""])
        d.append(["Annual average",SNB_AVG_EUR_CHF,""])
        d.append([f"31.12.{year}",YE_EUR_CHF,""])
        d.append([]); d.append(["Source","Swiss National Bank",""])
        d.append(["","https://data.snb.ch/en/topics/ziredev/cube/devkum",""])
        for col,w in zip("ABCD",[12,32,26,44]): d.column_dimensions[col].width=w
        hdr(d,4); styled(d)
    D=["ID","Date","Refined_Category_XZ","Category","Amount","Currency","EUR","CHF","Included_in_summary","Exclusion_reason","Source","Reference","Note"]
    def det(name,df):
        s=wb.create_sheet(name); s.append(D); hdr(s,len(D))
        for _,rw in df.iterrows():
            s.append([rw["ID"],rw["Date"].to_pydatetime() if pd.notna(rw["Date"]) else None,
                      rw.get("Refined_Category_XZ"),rw["Category"],
                      rw["Amount"],rw["Currency"],rw["Value"],rw.get("CHF"),rw["Included_in_summary"],
                      rw["Exclusion_reason"],rw["Source"],rw["Reference"],rw["Note"]])
            if rw["Included_in_summary"]=="No":
                for k in range(1,len(D)+1): s.cell(row=s.max_row,column=k).fill=GREY
        s.freeze_panes="A2"
        for col,w in zip("ABCDEFGHIJKLM",[30,19,28,28,13,9,12,12,18,58,12,26,42]): s.column_dimensions[col].width=w
        styled(s)
        for rr in s.iter_rows(min_row=2,min_col=2,max_col=2):
            for cell in rr: cell.number_format="yyyy-mm-dd"
    def summ(name,ser,serc=None):
        s=wb.create_sheet(name); s.append(["Refined_Category_XZ",f"Sum of {c['ccy']}","Sum of CHF"]); hdr(s,3)
        for k,v in ser.items(): s.append([k,v,(serc.get(k) if serc is not None else None)])
        s.append(["Grand Total",round(ser.sum(),2),round(serc.sum(),2) if serc is not None else None])
        for k in (1,2,3):
            cc2=s.cell(row=s.max_row,column=k); cc2.font=SEC; cc2.border=B_TOTAL
        s.column_dimensions["A"].width=36; s.column_dimensions["B"].width=16; s.column_dimensions["C"].width=16
        styled(s)
    det(f"{year} income",led[led.Side=="income"]);   summ(f"{year} income summary",IS,ISC)
    det(f"{year} expense",led[led.Side=="expense"]); summ(f"{year} expense summary",ES,ESC)
    if c["mode"]=="historical":
        for raw in c["raw"]:
            if raw in src.sheetnames:
                dd=wb.create_sheet("Raiffeisen" if raw=="Raiffreisen" else raw)
                for row in src[raw].iter_rows(values_only=True): dd.append(list(row))
                hdr(dd,dd.max_column); styled(dd); dd.freeze_panes="A2"
    else:
        sp,ra=raws
        s=wb.create_sheet("stripe_payments"); s.append(list(sp.columns)); hdr(s,len(sp.columns))
        for _,rw in sp.iterrows(): s.append([None if pd.isna(v) else v for v in rw.tolist()])
        hdr(s,len(sp.columns)); styled(s); s.freeze_panes="A2"
        s=wb.create_sheet("Raiffeisen"); s.append(list(ra.columns)); hdr(s,len(ra.columns))
        for _,rw in ra.iterrows(): s.append([None if pd.isna(v) else v for v in rw.tolist()])
        hdr(s,len(ra.columns)); styled(s); s.freeze_panes="A2"
    # summary
    s=wb.create_sheet(f"{year} summary")
    def ln(a=None,b=None,cc=None,bold=False,kind=None):
        """kind: None | 'hdr' (bold + rule under) | 'total' (bold + rules above and below)
                 | 'title' (22pt section heading)"""
        s.append([a,b,cc]); r=s.max_row
        for k in range(1,4):
            c=s.cell(row=r,column=k); c.font=BODY
            if k>1 and isinstance(c.value,(int,float)): c.number_format=ACC
        if kind=="title":
            s.cell(row=r,column=1).font=TITLE
        elif kind=="hdr":
            for k in range(1,4):
                c=s.cell(row=r,column=k); c.font=SEC; c.border=B_UNDER
                if k>1: c.alignment=RIGHT
        elif kind=="total" or bold:
            for k in range(1,4):
                c=s.cell(row=r,column=k); c.font=SEC
                if kind=="total": c.border=B_TOTAL
    s["A1"]=("Départ de Sentier\nDorfsteig 8\n5223 Riniken AG\nSwitzerland\n"
             "https://d-d-s.ch\ninfo@d-d-s.ch")
    s["A1"].font=ADDR; s["A1"].alignment=Alignment(wrap_text=True,vertical="top")
    s.row_dimensions[1].height=90
    ln(f"Income statement — year ended 31 December {year}",None,None,kind="title"); ln()
    ln("Income",f"Amount ({c['ccy']})","Amount (CHF)",kind="hdr")
    for k,v in sorted(GI.items()): ln(k,v,GIC.get(k))
    ln("Total income",round(sum(GI.values()),2),round(sum(GIC.values()),2),kind="total"); ln()
    ln("Expense",f"Amount ({c['ccy']})","Amount (CHF)",kind="hdr")
    for k,v in sorted(GE.items()): ln(k,v,GEC.get(k))
    ln("Total expense",round(sum(GE.values()),2),round(sum(GEC.values()),2),kind="total"); ln()
    ln("Operating result",op,opc,kind="total")
    ln("Foreign-exchange difference",None,fx)
    ln(f"Net result for {year}",None,round(opc+fx,2),kind="total"); ln()
    ln(f"Balance sheet as of 31 December {year}",None,None,kind="title"); ln()
    ln("Assets",None,None,True)
    ln("Cash and cash equivalents","Amount (EUR)","Amount (CHF)",kind="hdr")
    for n,e,v in close_cash: ln(n,e,round(v,2) if v is not None else None)
    ln("Total cash and cash equivalents",None,tot_cash,kind="total")
    ln("Accounts receivable","Amount (EUR)","Amount (CHF)",kind="hdr")
    for n,e,v in (c["receivable"]+recv): ln(n,round(e,2),round(v,2))
    ln("Total accounts receivable",None,tot_recv,kind="total")
    ln("Total assets",None,assets,kind="total"); ln()
    ln("Liabilities",None,"Amount (CHF)",kind="hdr")
    for n,v in (c["liabilities"]+liab): ln(n,None,round(v,2))
    if not (c["liabilities"]+liab): ln("None — no payables at the balance-sheet date",None,0.00)
    ln("Total liabilities",None,tot_liab,kind="total"); ln()
    ln("Equity",None,"Amount (CHF)",kind="hdr")
    ln(f"Accumulated surplus at 01.01.{year}",None,round(c["open_eq"],2))
    ln(f"Net result for {year}",None,round(opc+fx,2))
    ln(f"Accumulated surplus at 31.12.{year}",None,equity,kind="total"); ln()
    ln("Cash and cash equivalents — reconciliation (CHF)",None,None,kind="title")
    ln("(Wise held in EUR; reporting currency CHF)")
    ln("Account","Amount (EUR)","Amount (CHF)",kind="hdr")
    if c["founding"]: ln("Cash at inception",None,0.00)
    else:
        for n,e,v in c["open_cash"]: ln(n,e,round(v,2) if v is not None else None)
    ln(f"Total cash at 01.01.{year}",None,round(c["open_eq"] if c["founding"] else sum(v for _,_,v in c["open_cash"] if v),2),kind="total")
    ln(f"Net result for {year}",None,round(opc+fx,2))
    if accr:
        ln("Movements not passing through the result:")
        for n,v in liab: ln(f"  {n}",None,round(v,2))
        for n,e,v in recv: ln(f"  less {n}",None,round(-v,2))
    ln(f"Total cash at 31.12.{year}",None,tot_cash,kind="total"); ln()
    ln(f"The following table provides an overview of balances for major activities in {year} and")
    ln("does not represent cash balances or balance sheet positions. Project grants are excluded:")
    ln("they reimburse eligible costs, so an annual net figure reflects the reimbursement rate and")
    ln("the flat overhead percentage rather than how the project performed. Their cumulative")
    ln("position against contract value is shown separately below.")
    ln(f"Summary of schools, conference and projects {year}",None,None,kind="title")
    ln("School/Conference",f"Amount ({c['ccy']})","Amount (CHF)",kind="hdr")
    for a in ACTIVITIES:
        n=GI.get(a,0)-GE.get(a,0)
        if abs(n)>0.005: ln(a,round(n,2),round(GIC.get(a,0)-GEC.get(a,0),2))
    if accr:
        ln()
        ln(f"Project grants — position at 31 December {year}",None,None,kind="hdr")
        ln("GreenGrocer (SERI 25.00414), contract to 31.08.2029",None,None)
        ln("  contract value (maximum)",None,591481.00)
        ln("  received to date (first instalment)",None,GG_CASH)
        ln("  recognised as income in the year",None,round(g["gg_total"],2))
        ln("  carried as deferred income",None,round(g["gg_def"],2))
        ln("ADEME (convention 2403D0042), contract to ~04.2027",None,None)
        ln("  maximum aid (70% of eligible costs)",None,round(182420.00*SNB_AVG_EUR_CHF,2))
        ln("  entitlement earned to date",None,round(g["ent"]*SNB_AVG_EUR_CHF,2))
        ln("  received in cash to date",None,round(ADEME_CASH*SNB_AVG_EUR_CHF,2))
        ln("  carried as accrued income receivable",None,round(g["acc"]*YE_EUR_CHF,2))
    # ---------------- Notes ----------------
    ln(); ln("Notes:",None,None,bold=True)
    n=0
    def note(head,*body):
        nonlocal n; n+=1
        ln(f"{n}. {head}")
        for line in body: ln(line)
    note("Foreign currency translation",
         f"Income and expenses in foreign currencies are translated into CHF using the "
         f"Swiss National Bank {year} "
         + ("monthly average rates" if (accr and MONTHLY_OK) else
            ("annual average rate" if year>=2023 else "monthly exchange rates"))
         + ", based on the value of the original currency."
         + (" Where a transaction carries an actual booked exchange rate (Wise, Stripe) that rate is"
            " used; where the amount is denominated in CHF the CHF figure is taken directly." if accr else ""))
    note("Cash balances",
         f"Cash balances in foreign currencies are translated into CHF at the rate applicable at the reporting "
         f"date (31.12.{year})." + ("" if c["founding"] else f" Opening balances use the rate as of 31.12.{year-1}."))
    note("Foreign exchange difference",
         "Foreign exchange differences arise from translating foreign-currency transactions and balances at",
         "different exchange rates and are recognised separately in the income statement.")
    note("Reporting currency",
         "The financial statements are prepared in Swiss francs (CHF). Amounts shown in other currencies are",
         "provided for information only." + (" During the year the association also held SGD and CAD balances; "
         "both opened and closed at nil." if accr else ""))
    if c["founding"]:
        note("Establishment of the association",
             "As the association commenced activities in 2022, opening cash balances were zero. The balance sheet",
             "recognises a receivable of CHF 2,504.40 for Brightcon income invoiced but not received at the year end.")
    if accr:
        note("Change of accounting basis (2023–2024 cash basis; 2025 accrual basis)",
             "The 2022 statement already recognised a receivable. The 2023 and 2024 statements carried no",
             "receivables or payables and were substantially cash-based. From 2025 the statement is prepared on an",
             "accrual basis: grant income is recognised as eligible costs are incurred, and 2025 costs settled in",
             "2026 are accrued. Because this affects comparability (Art. 958c CO), the cash reconciliation above",
             "bridges the net result to the movement in cash; the non-cash lines are the deferred income, accruals",
             "and receivable introduced by the accrual basis.")
        note("Project grants",
             f"GreenGrocer (SERI contract 25.00414, Horizon Europe 101182025): CHF {GG_CASH:,.2f} received as the",
             "first of three instalments for the period 01.09.2025–31.08.2029. Recognised as income only to the",
             f"extent of eligible costs incurred (CHF {g['gg_total']:,.2f}); the balance of CHF {g['gg_def']:,.2f} is",
             "carried as deferred income. In accordance with contract clause 3.2 the funds are reported separately.",
             f"ADEME (convention 2403D0042): reimburses 70% of eligible costs. The entitlement of EUR {g['ent']:,.2f}",
             f"earned in excess of the EUR {ADEME_CASH:,.2f} received is carried as an accrued income receivable.")
    if accr:
        note("Prior-period presentation (2022 to 2023)",
             "The 2022 statement closed with accumulated surplus of CHF 10,077.12, comprising cash of",
             "CHF 7,572.72 and a Brightcon receivable of CHF 2,504.40. The 2023 statement opened at",
             "CHF 7,572.72, i.e. from cash rather than from the accumulated surplus, so the receivable of",
             "CHF 2,504.40 was recognised as income in 2022 and again on receipt in 2023. The 2022-2024",
             "statements were approved by the general assembly and are not restated. The equity chain from",
             "1 January 2023 onward is unbroken. From 2025 each statement opens at the prior year's closing",
             "equity, and this is verified before issue.")
    if accr:
        note("Accrued income",
             f"Three Brightcon 2025 registrations totalling EUR {sum(a for _,a in BRIGHTCON_2026):,.2f} were",
             "invoiced in 2025 but settled in 2026 (" +
             ", ".join(f"EUR {a:,.2f} on {d}" for d,a in BRIGHTCON_2026) + ").",
             "The conference was held in October 2025, so the income is recognised in 2025 with a matching",
             "receivable at the balance-sheet date. These receipts must be booked against the receivable in",
             "2026, not recognised again as 2026 income.")
    note("Refunds and excluded items",
         "All transactions are retained in the detail sheets; none are deleted. Refunds of expenses are booked as",
         "negative contra-entries within their original expense category, never as income, so income and expenses",
         "are not offset (Art. 958c CO) and the ledger stays complete (Art. 957a CO). Internal transfers and",
         "failed or cancelled transactions are retained but excluded from the summary.")
    if accr:
        note("Net assets",
             f"Net assets at 31 December {year} are negative by CHF {abs(equity):,.2f}. This reflects the accrual",
             f"presentation rather than a shortage of funds: cash of CHF {tot_cash:,.2f} is held, and the largest",
             f"liability — deferred GreenGrocer income of CHF {g['gg_def']:,.2f} — is future revenue that will be",
             "released as project costs are incurred, not a debt falling due.")
        note("Assumed tax and fine for 2023",
             "CHF 775 (CHF 425 assumed tax, CHF 350 late-payment fine), both paid, relating to 2023, for which no",
             "tax return was filed. The underlying tax amount remains an estimate pending clarification with the",
             "Aargau tax authority on whether retroactive 2023 filing is possible. The association has filed",
             "compliantly since 2024.")
    for col,w in zip("ABC",[58,16,16]): s.column_dimensions[col].width=w
    s.sheet_view.showGridLines=False
    for row in s.iter_rows(min_row=2):
        for cell in row:
            if cell.font.name != FONT:
                cell.font=Font(name=FONT,size=cell.font.size or SZ,bold=cell.font.bold)
    # ---------- RULE 10: the equity chain must articulate ----------
    lhs=round(c["open_eq"]+opc+fx,2); breaks=[]
    if abs(lhs-equity)>=0.05:
        breaks.append(f"opening {c['open_eq']:,.2f} + operating {opc:,.2f} + FX {fx:,.2f} = {lhs:,.2f}, "
                      f"but the balance sheet shows equity of {equity:,.2f} "
                      f"(difference {equity-lhs:+,.2f})")
    prior=CLOSING.get(year-1)
    if prior is not None and abs(prior-c["open_eq"])>0.01:
        breaks.append(f"opens at {c['open_eq']:,.2f} but {year-1} closed at {prior:,.2f} "
                      f"(difference {c['open_eq']-prior:+,.2f}) — a statement must open at the prior year's "
                      f"CLOSING EQUITY, never at prior-year cash")
    if breaks:
        msg=f"RULE 10 — equity chain break in {year}:\n    " + "\n    ".join(breaks)
        if c["mode"]=="sources":
            raise AssertionError(msg+"\n  A statement that does not articulate must not be issued.")
        print(f"  !! {msg}\n     (published statement, approved by the GA — disclose, do not restate; "
              f"see project instructions §11)")
    CLOSING[year]=equity
    path=f"{OUT}/{year}_DdS_Financial_Balance_Sheet.xlsx"
    wb.save(path)
    print(f"{year}: income {sum(GI.values()):>12,.2f} {c['ccy']} | expense {sum(GE.values()):>12,.2f} | "
          f"operating CHF {opc:>10,.2f} | FX {fx:>9,.2f} | net {opc+fx:>10,.2f} | equity {equity:>10,.2f}")
    return path

## Cell 10 — Generate

Runs all four years in sequence. Order matters: `CLOSING` accumulates each year's closing equity so
the next year's continuity check has something to compare against.

Each line prints income, expense, operating result, FX difference, net result and closing equity.
`RULE 10` warnings appear above the year they concern.

In [5]:
for y in (2022, 2023, 2024, 2025):
    write(y)

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Slicer List extension is not supported and will be removed
  warn(msg)


  !! RULE 10 — equity chain break in 2022:
    opening 0.00 + operating 6,026.90 + FX 1,545.81 = 7,572.71, but the balance sheet shows equity of 10,077.12 (difference +2,504.41)
     (published statement, approved by the GA — disclose, do not restate; see project instructions §11)
2022: income    38,331.27 CHF | expense    32,304.37 | operating CHF   6,026.90 | FX  1,545.81 | net   7,572.71 | equity  10,077.12


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Slicer List extension is not supported and will be removed
  warn(msg)


  !! RULE 10 — equity chain break in 2023:
    opens at 7,572.72 but 2022 closed at 10,077.12 (difference -2,504.40) — a statement must open at the prior year's CLOSING EQUITY, never at prior-year cash
     (published statement, approved by the GA — disclose, do not restate; see project instructions §11)
2023: income    44,386.97 EUR | expense    42,257.41 | operating CHF   2,067.22 | FX   -588.42 | net   1,478.80 | equity   9,051.52


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Slicer List extension is not supported and will be removed
  warn(msg)


2024: income   124,323.13 EUR | expense   121,651.72 | operating CHF   2,544.16 | FX -2,196.66 | net     347.50 | equity   9,399.02


2025: income   193,903.47 EUR | expense   208,076.61 | operating CHF -12,913.11 | FX  1,993.85 | net -10,919.26 | equity  -1,520.24


## Notes on the results

**2022** — the founding year: opening cash was zero, and the balance sheet carried a Brightcon
receivable of CHF 2,504.40, so closing equity is CHF 10,077.12 rather than the CHF 7,572.72 of cash.
Note that the published 2023 statement opens from **cash** (7,572.72) rather than from that equity,
so there is a CHF 2,504.40 break in the association's own historical chain. Worth raising with the
auditor.

**2023 and 2024** — reproduce the published statements. The only substantive differences are the
rule 9 refund reclassifications, which move value between the operating result and the FX difference
while leaving **net result and closing equity unchanged**.

**2025** — the first year on a full accrual basis. Cash rose by CHF 306,330 while the result was
negative; the cash reconciliation on the summary tab shows why (grant received in advance, accrued
costs not yet paid, ADEME income not yet received).